In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dask.dataframe as dd
import pandas as pd

# Get barcode images
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import holoviews as hv
from holoviews.operation.datashader import rasterize
import hvplot.xarray # noqa
import panel as pn
import fsspec

pn.extension('tabulator')
# hv.extension('bokeh')
hv.extension('bokeh', 'matplotlib')
# %matplotlib inline

In [ ]:
def writedir(directory,overwrite=False):
    if overwrite:
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory)
    else:
        if not os.path.exists(directory):
            os.makedirs(directory)

import os
import shutil
import dask
import time
import h5py
import threading
import dask.distributed

from time import sleep
from dask.distributed import Client,progress
from dask.distributed.diagnostics.plugin import WorkerPlugin
from dask_jobqueue import SLURMCluster

# from IPython.core.display import display, HTML
from IPython.display import display, HTML

## Hacky memory trim
import ctypes
import gc

def trim_memory() -> int:
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

class dask_controller: #adapted from Charles' code
    def __init__(self,n_workers=6,n_workers_min=6,local=True,queue="short",death_timeout=60.,\
                 walltime='01:00:00',cores=1,processes=1,memory='6GB',\
                 working_directory="./",job_extra_directives=[]):
        self.local = local
        self.n_workers = n_workers
        self.n_workers_min = n_workers_min
        self.walltime = walltime
        self.queue = queue
        self.death_timeout = death_timeout
        self.processes = processes
        self.memory = memory
        self.cores = cores
        self.working_directory = working_directory
        self.job_extra_directives = job_extra_directives

        self.futures = {}

        split_walltime = walltime.split(":")
        wall_hrs,wall_mins,wall_secs = tuple(split_walltime)
        ttl_wall_mins = (60*int(wall_hrs)) + int(wall_mins)

        if ttl_wall_mins < 10:
            raise ValueError("Walltime must be at least 10 mins long!")

        self.lifetime = str(max(ttl_wall_mins-10,10)) + "m"
        print(self.lifetime)
        print(walltime)
        self.worker_extra_args = ["--lifetime", self.lifetime, "--lifetime-stagger", "5m"]

        writedir(working_directory,overwrite=False)

    def startdask(self):
        if self.local:
            self.daskclient = Client()
            self.daskclient.cluster.scale(self.n_workers)
        else:
            self.daskcluster = SLURMCluster(n_workers=self.n_workers_min,queue=self.queue,death_timeout=self.death_timeout,walltime=self.walltime,\
                                   processes=self.processes,memory=self.memory,cores=self.cores,local_directory=self.working_directory,\
                                log_directory=self.working_directory,worker_extra_args=self.worker_extra_args,job_extra_directives=self.job_extra_directives)
            self.daskcluster.adapt(minimum=self.n_workers_min, maximum=self.n_workers,\
                                   interval="1m",wait_count=10)
            self.daskclient = Client(self.daskcluster)

    def shutdown(self, delete_files=True):
        self.reset_worker_memory()
        if not self.local:
            self.daskcluster.close()
        if delete_files:
            for item in os.listdir(self.working_directory):
                if "worker-" in item or "slurm-" in item or ".lock" in item:
                    path = "./" + item
                    if os.path.isfile(path):
                        os.remove(path)
                    elif os.path.isdir(path):
                        shutil.rmtree(path)

    def printprogress(self):
        complete = len([item for item in self.futures if item.status=="finished"])
        print(str(complete) + "/" + str(len(self.futures)))

    def displaydashboard(self):
        link = self.daskcluster.dashboard_link
        display(HTML('<a href="' + link +'">Dashboard</a>'))

    def mapfovs(self,function,fov_list,retries=0):
        self.function = function
        self.retries = retries
        def mapallfovs(fov_number,function=function):
            function(fov_number)
        self.futures = {}
        for fov in fov_list:
            future = self.daskclient.submit(mapallfovs,fov,retries=retries)
            self.futures[fov] = future

    def retry_failed(self):
        self.failed_fovs = [fov for fov,future in self.futures.items() if future.status != 'finished']
        out = self.daskclient.restart()
        self.mapfovs(self.function,self.failed_fovs,retries=self.retries)

    def retry_processing(self):
        self.proc_fovs = [fov for fov,future in self.futures.items() if future.status == 'pending']
        out = self.daskclient.restart()
        self.mapfovs(self.function,self.proc_fovs,retries=self.retries)

    def restart(self):
        self.daskclient.restart()
        with self.failure_lock:
            self.failure_counter_variable.set(0)

    def reset_worker_memory(self):
        self.daskclient.cancel([val for key,val in self.futures.items()])
        self.daskclient.run(gc.collect)
        self.daskclient.run(trim_memory)
        print("Done.")

class hdf5lock:
    def __init__(self,filepath,updateperiod=0.1):
        self.filepath = filepath
        self.lockfile = filepath + ".lock"
        self.updateperiod = updateperiod

    def _lock(self):
        while True:
            if not os.path.exists(self.lockfile):
                open(self.lockfile,'w').close()
                break
            sleep(self.updateperiod)

    def _apply_fn(self,function,iomode,*args,**kwargs):
        try:
            fn_output = function(self.filepath,iomode,*args,**kwargs)
            os.remove(self.lockfile)
            return fn_output
        except:
            os.remove(self.lockfile)
            raise

    def lockedfn(self,function,iomode,*args,**kwargs):
        self._lock()
        fn_output = self._apply_fn(function,iomode,*args,**kwargs)
        return fn_output

def transferjob(sourcedir,targetdir,single_file=False):
    mkdircmd = "mkdir -p '" + targetdir + "'"
    if single_file:
        rsynccmd = "rsync -r '" + sourcedir + "' '" + targetdir + "'"
    else:
        rsynccmd = "rsync -r '" + sourcedir + "/' '" + targetdir + "'"
    wrapcmd = mkdircmd + " && " + rsynccmd
    cmd = "sbatch -p transfer -t 0-12:00 --wrap=\"" + wrapcmd + "\""
    print(cmd)
    os.system(cmd)

In [ ]:
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/lag36/scratch/lag36/dask"

dask_controller_instance = dask_controller(
    walltime="3:00:00",
    local=False,
    n_workers=10,
    n_workers_min=10,#200,
    memory="20GB",
    working_directory=dask_wd,
)
dask_controller_instance.startdask()
dask_controller_instance.displaydashboard()

In [ ]:
def read_barcode_output_hdf5(filename):
    df_bar_fish = pd.read_hdf(filename) #read in the barcode fish data
    with pd.HDFStore(filename,'r') as store:
        df_bar_fish_metadata = store.get_storer('barcodes').attrs.metadata

    return df_bar_fish, df_bar_fish_metadata

def print_stats(df_bar_fish, df_bar_fish_metadata):
    print("Total trenches:           ", df_bar_fish_metadata['Total Trenches'])
    print("Total Trenches With Cells: ", df_bar_fish_metadata['Total Trenches With Cells'])
    print("Removed Bits:             ", df_bar_fish_metadata['Removed Bits'])
    print("Hamming Threshold':      ", df_bar_fish_metadata['Hamming Threshold'])
    print("Called:                   ", len(df_bar_fish))
    print("Called percentage:        ", len(df_bar_fish)/df_bar_fish_metadata['Total Trenches With Cells'])

In [ ]:
headpath = '/home/lag36/scratch/lag36/Ecoli/'
filename = headpath + '/2022-01-18_lDE20_Final_5/Barcodes/barcode_output_df.hdf5'
df_bar_fish, df_bar_fish_metadata = read_barcode_output_hdf5(filename)
print_stats(df_bar_fish, df_bar_fish_metadata)

In [ ]:
headpath = '/home/lag36/scratch/lag36/Ecoli/'
filename = headpath + '/2022-01-20_lDE20_Final_6/Barcodes/barcode_output_df.hdf5'
df_bar_fish, df_bar_fish_metadata = read_barcode_output_hdf5(filename)
print_stats(df_bar_fish, df_bar_fish_metadata)

In [ ]:
headpath = '/home/lag36/scratch/lag36/Ecoli/'
filename = headpath + '/2023-01-02_lDE20_Run_8/Barcodes/barcode_output_df.hdf5'
df_bar_fish, df_bar_fish_metadata = read_barcode_output_hdf5(filename)
print_stats(df_bar_fish, df_bar_fish_metadata)

In [ ]:
headpath = '/home/lag36/scratch/lag36/Ecoli/'
filename = headpath + '/2023-01-14_lDE20_Run_9/Barcodes/barcode_output_df.hdf5'
df_bar_fish, df_bar_fish_metadata = read_barcode_output_hdf5(filename)
print_stats(df_bar_fish, df_bar_fish_metadata)

In [ ]:
headpath = '/home/lag36/scratch/lag36/'
filename_lLAG10 = headpath + '/2024-12-05_lLAG10_MBM_Run-2_3-Fiducials_Temp-Fixed/Barcodes/barcode_output_df_fiducials_hamming-2_rmBits-None.hdf5'
df_bar_fish_lLAG10, df_bar_fish_metadata_lLAG10 = read_barcode_output_hdf5(filename_lLAG10)

In [ ]:
print_stats(df_bar_fish_lLAG10, df_bar_fish_metadata_lLAG10)

In [ ]:
headpath = '/home/lag36/scratch/lag36/'
filename_lLAG8 = headpath + '/2024-08-16_lLAG8_Run-1/Barcodes/barcode_output_df.hdf5'
df_bar_fish_lLAG8, df_bar_fish_metadata_lLAG8 = read_barcode_output_hdf5(filename_lLAG8)

In [ ]:
print_stats(df_bar_fish_lLAG8, df_bar_fish_metadata_lLAG8)

### Draw histograms and check non-called trenches

In [ ]:
# Get percentiles
headpath = '/home/lag36/scratch/lag36/Ecoli/'
filename = headpath + '/2022-01-18_lDE20_Final_5/Barcodes/barcode_output_df.hdf5'
df_bar_fish, df_bar_fish_metadata = read_barcode_output_hdf5(filename)
print_stats(df_bar_fish, df_bar_fish_metadata)

In [ ]:

def load_percentiles(filename, load_hdf5, filename_save_hdf5, channel_cols):
    '''
    Load the percentiles from the parquet files. If load_hdf5 is True, then the file is loaded from the HDF5 file.
        filename (str): the filename of the parquet file
        load_hdf5 (bool): whether to load the data from the HDF5 file
        filename_save_hdf5 (str): the filename to save the data to the HDF5 file
    '''

    if load_hdf5:
        df_percentiles = pd.read_hdf(filename)
    else:
        df_percentiles = (dd.read_parquet(filename,
                                        engine="pyarrow",
                                        calculate_divisions=True,
                                        columns=['timepoints', 'trenchid'] + channel_cols)
                            .astype({'trenchid':'int32', 'timepoints':'int16'})
                            .compute()
        )
        if filename_save_hdf5 is not None:
            df_percentiles.to_hdf(filename_save_hdf5, key='data')
    return df_percentiles

In [ ]:
filename = headpath + '/2022-01-20_lDE20_Final_6/Barcodes/percentiles'
filename_hdf5 = headpath + '/2022-01-20_lDE20_Final_6/Barcodes/percentiles.hdf5'

In [ ]:
df_percentiles = load_percentiles(filename,
                                  load_hdf5=False,
                                  filename_save_hdf5=filename_hdf5,
                                  channel_cols=['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile'])

In [ ]:
from holoviews.operation import histogram

In [ ]:
def generate_percentile_histograms(df_percentiles: pd.DataFrame, 
                                   sample:int,
                                   channel_cols: list,
                                   bin_ranges: dict,
                                   y_lims: dict):
    


    trench_ids = df_percentiles['trenchid'].unique()
    trench_ids = np.random.choice(trench_ids, sample, replace=False)

    df_percentiles_sample = (df_percentiles
                    .loc[df_percentiles['trenchid'].isin(trench_ids)]
                    .reset_index()
                    .loc[:,['timepoints', 'trenchid'] + channel_cols]
                    )

    data_hists = hv.Dataset(df_percentiles_sample)

    hists = {}
    for ch in channel_cols:
        hists[ch] = histogram(
            data_hists.to(hv.Points, kdims=['trenchid'] + [ch], groupby=['timepoints']),
            num_bins = 500,
            bin_range = bin_ranges[ch],
            dimension=ch,
        ).opts(
            ylim = y_lims[ch],
            width = 300,
            height=150,
        )
    
    return hists

def generate_bit_threshold_lines(bit_thresh_list: list,
                                 channel_cols: list):
    
    assert len(bit_thresh_list) % len(channel_cols) == 0
    bits_per_channel = len(bit_thresh_list) // len(channel_cols)

    bit_thresholds = {channel_cols[i]: bit_thresh_list[i*bits_per_channel:(i+1)*bits_per_channel] for i in range(len(channel_cols))}
    bit_thresholds['timepoints'] = np.arange(bits_per_channel, dtype=np.int8)
    bit_thresholds = hv.Dataset(pd.DataFrame(bit_thresholds))

    lines = {}
    for ch in channel_cols:
        lines[ch] = bit_thresholds.to(hv.VLines, kdims=[ch], groupby=['timepoints']).opts(
            line_width=1,
        )
    return lines

In [ ]:
sample = 200000
channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']

bin_ranges = {}
bin_ranges['RFP 98th Percentile'] = (0, 6000)
bin_ranges['Cy5 98th Percentile'] = (0, 50000)
bin_ranges['Cy7 98th Percentile'] = (0, 4000)

y_lims = {}
y_lims['RFP 98th Percentile'] = (0, 3000)
y_lims['Cy5 98th Percentile'] = (0, 3000)
y_lims['Cy7 98th Percentile'] = (0, 3000)


hists = generate_percentile_histograms(df_percentiles,
                                       sample = 200000,
                                       channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile'],
                                       bin_ranges = bin_ranges,
                                       y_lims = y_lims)

In [ ]:
channel_cols = ['RFP 98th Percentile', 'Cy5 98th Percentile', 'Cy7 98th Percentile']
bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']

lines = generate_bit_threshold_lines(bit_thresh_list, channel_cols)

In [ ]:
hist_grids = {ch: hv.NdLayout(hists[ch] * lines[ch]).cols(2) for ch in channel_cols}

In [ ]:
import panel

In [ ]:
super_layout = panel.Row(
    hist_grids['RFP 98th Percentile'],
    hist_grids['Cy5 98th Percentile'],
    hist_grids['Cy7 98th Percentile'],
)

In [ ]:
super_layout

In [ ]:
import panel

In [ ]:
hv.NdLayout(hist_RFP*lines_RFP*lines_trench_RFP).opts(shared_axes=False).cols(2)

In [ ]:
hists['RFP 98th Percentile']*lines['RFP 98th Percentile'] + hists['Cy5 98th Percentile']*lines['Cy5 98th Percentile'] + hists['Cy7 98th Percentile']*lines['Cy7 98th Percentile']

In [ ]:
bit_thresholds.data

In [ ]:
bit_thresh_list = df_bar_fish_metadata['Bit Threshold List']

bit_thresholds = pd.DataFrame({
    'timepoints': np.arange(15, dtype=np.int8),
    'RFP 98th Percentile': bit_thresh_list[:15],
    'Cy5 98th Percentile': bit_thresh_list[15:],
})